# Exploratory Data Analysis — Turizmu Timor-Leste

**Capstone Project | Mining and Exploring | S1 Siencia Komputador | DIT 2026**

Notebook ida-ne'e halao **EDA** tuir *Manual Capstone Project seksaun 12*:

| No. | Pontu Manual seksaun 12 | Seksaun iha notebook |
|-----|--------------------------|----------------------|
| 1 | Descriptive statistics (count, mean, median, min, max, std) | §3 |
| 2 | Distribution analysis (histogram, boxplot, density) | §4 |
| 3 | Trend analysis (line chart tuir tempu) | §5 |
| 4 | Relationship analysis (correlation, scatterplot) | §6 |
| 5 | Category analysis (frequency no ranking) | §7 |
| 6 | Geographical/spatial analysis | §8 |
| 7 | Insight prinsipal no pergunta ba data mining | §9 |

> **Python 3.11 obrigatóriu.** Se kernel la'ós 3.11, troka iha *Kernel → Change Kernel*.
> Instala kernel: `py -3.11 -m ipykernel install --user --name turismo311`

In [ ]:
import sys
from pathlib import Path

# Hatama folder src/ ba path atu bele import modulu pipeline
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
assert sys.version_info[:2] == (3, 11), "Projetu ne'e presiza Python 3.11!"

## 1. Data Collection

Fonte dadus: **snapshot JSON Google Maps reviews** (Apify scraper).
Grupu seidauk iha API key, tanba ne'e uza modu *batch snapshot* — dokumenta iha
`src/config.py` (`DATA_SOURCE`) no README seksaun 2.

In [ ]:
from config import DATA_SOURCE, resolve_raw_dataset
from data_collection import collect

print("Dataset:", resolve_raw_dataset())
for k, v in DATA_SOURCE.items():
    print(f"  {k:28s}: {v}")

raw = collect()
print()
print("Baris raw :", len(raw))
print("Lokal     :", raw["place_id"].nunique())
raw.head(3)

## 2. Preprocessing (Manual seksaun 11)

Etapa: Cleaning → Missing Values → Transformation → Outliers → Text →
Feature Engineering → Train/Test Split.

In [ ]:
from preprocessing import preprocess

df, prep_meta = preprocess(raw)

for etapa, info in prep_meta.items():
    print(f"--- {etapa.upper()} ---")
    if isinstance(info, dict):
        for k, v in info.items():
            print(f"    {k}: {v}")
    print()

## 3. Descriptive Statistics

*Manual seksaun 12, pontu 1: count, mean, median, minimum, maximum, standard deviation.*

In [ ]:
from category_classifier import apply_category_classification
from sentiment_analysis import apply_sentiment_analysis
from eda import descriptive_statistics

df = apply_category_classification(df)
df = apply_sentiment_analysis(df)

stats = descriptive_statistics(df)
stats

In [ ]:
print("Rezumu jeral")
print(f"  Lokal turizmu       : {df['place_id'].nunique()}")
print(f"  Total review        : {len(df)}")
print(f"  Review ho teksu     : {int(df['has_text'].sum())}")
print(f"  Review rating-only  : {int((~df['has_text']).sum())}")
print(f"  Rating mediu        : {df['review_stars'].mean():.2f}")
print(f"  Rating median       : {df['review_stars'].median():.1f}")
print(f"  Periodu review      : {df['review_date_parsed'].min()} — {df['review_date_parsed'].max()}")

## 4. Distribution Analysis

*Manual seksaun 12, pontu 2: histogram, boxplot ka density.*

In [ ]:
from eda import plot_rating_distribution, plot_text_length_distribution
from IPython.display import Image, display

display(Image(plot_rating_distribution(df)))
display(Image(plot_text_length_distribution(df)))

In [ ]:
print("Distribuisaun rating:")
print(df["review_stars"].value_counts().sort_index().to_string())
print()
print("Distribuisaun sentiment:")
print(df["sentiment"].value_counts().to_string())
print()
print("Fonte label sentiment:")
print(df["sentiment_source"].value_counts().to_string())

## 5. Trend Analysis

*Manual seksaun 12, pontu 3: line chart tuir tempu.*

In [ ]:
from eda import plot_trend

display(Image(plot_trend(df)))

monthly = (df.dropna(subset=["review_date_parsed"])
             .assign(fulan=lambda d: d["review_date_parsed"].dt.tz_localize(None)
                                        .dt.to_period("M").astype(str))
             .groupby("fulan").size())
print("Fulan ho review barak liu:")
print(monthly.sort_values(ascending=False).head(10).to_string())

## 6. Relationship Analysis

*Manual seksaun 12, pontu 4: correlation, scatterplot/comparison.*

In [ ]:
from eda import plot_correlation

fig_path, corr = plot_correlation(df)
display(Image(fig_path))
corr

### Interpretasaun korelasaun

Observa relasaun sira ne'ebé forte liu (|r| > 0.3) no komenta iha relatóriu
Chapter 5. Atensaun: korelasaun **la'ós** kauzalidade.

## 7. Category Analysis

*Manual seksaun 12, pontu 5: frequency no ranking.*

Kategoria hat (pedidu dosente): **Sejarah · Agama · Tradisional · Moderen**.
Label mai husi anotasaun manual (`data/place_category_labels.csv`).

In [ ]:
from eda import plot_category_sentiment, plot_category_ranking

display(Image(plot_category_ranking(df)))

fig_path, cross, tidy = plot_category_sentiment(df)
display(Image(fig_path))
cross

In [ ]:
# PERGUNTA DOSENTE: sentiment positivu iha kategoria ida-ne'ebe?
pct = (df.assign(is_pos=df["sentiment"].eq("Positive"))
         .groupby(df["category"].astype(str))["is_pos"].mean() * 100).round(1)
pct = pct.sort_values(ascending=False)
print("% review positivu kada kategoria:")
print(pct.to_string())
print()
print(f"Aas liu  : {pct.index[0]} ({pct.iloc[0]}%)")
print(f"Kraik liu: {pct.index[-1]} ({pct.iloc[-1]}%)")

## 8. Geographical / Spatial Analysis

*Manual seksaun 12, pontu 6: analiza jeografika se dataset iha koordenada.*

In [ ]:
from eda import plot_geographic

display(Image(plot_geographic(df)))

print("Review kada cidade/munisipiu (top 10):")
print(df["city"].value_counts().head(10).to_string())

## 9. Insight prinsipál no pergunta ba Data Mining

*Manual seksaun 12, pontu 7.*

In [ ]:
from eda import run_eda

eda_result = run_eda(df, make_figures=False)

print("PERGUNTA NE'EBE DATA MINING SEI RESPONDE:")
for i, q in enumerate(eda_result["key_questions_for_mining"], 1):
    print(f"  {i}. {q}")

print()
print("Rating mediu kada kategoria:", eda_result["avg_rating_by_category"])
print("Lokal ho review barak liu :", list(eda_result["top_places_by_reviews"].items())[:5])

## 10. Pasu tuir mai

EDA ne'e hatudu katak dadus prontu ba **Data Mining** (Manual seksaun 13).
Atu halao pipeline kompletu (clustering + klasifikasaun + evaluation + export
Power BI):

```powershell
py -3.11 src\pipeline.py
```

Depois, loke dashboard:

```powershell
py -3.11 -m streamlit run dashboard\app.py
```